In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve
from sklearn.base import BaseEstimator
from sklearn.calibration import calibration_curve

# sksurv 库 [修复点：补全 brier_score]
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, integrated_brier_score, brier_score

# 设置
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 0. 定义 XGBoost 适配器
# ==========================================
class XGBoostSurvival(xgb.XGBRegressor):
    def fit(self, X, y, **kwargs):

        y_xgb = np.where(y['event'], y['time'], -y['time'])
        super().fit(X, y_xgb, **kwargs)
        return self
    def predict(self, X):
        return super().predict(X)

def get_60m_survival_status(df, time_col='SurvivalMonths', event_col='Status', time_point=60):

    time = df[time_col].values
    event = df[event_col].values
    
    status = np.zeros(len(df), dtype=int)
    
    # 情况 1: 随访时间 >= 60 个月
    mask_long_followup = time >= time_point
    status[mask_long_followup] = np.where(
        (event[mask_long_followup] == 1) & (time[mask_long_followup] <= time_point),
        1,  # 60 个月内死亡
        0   # 60 个月时存活
    )
    
    # 情况 2: 随访时间 < 60 个月且已死亡
    mask_early_death = (time < time_point) & (event == 1)
    status[mask_early_death] = 1
    
    # 情况 3: 随访时间 < 60 个月且未死亡 (删失)
    # 按随访时长比例分配：随访越接近 60 个月，越可能归类为"存活"
    mask_censored = (time < time_point) & (event == 0)
    followup_ratio = time[mask_censored] / time_point
    status[mask_censored] = np.where(followup_ratio > 0.5, 0, 1)
    
    return status

# ==========================================
# 1. 配置与数据加载
# ==========================================
# !!! 请修改文件路径 !!!
path_seer = "" 
path_china = ""

NUMERIC_FEATURES = ["Age", "Tumor_Size"]
CATEGORICAL_FEATURES = [
    "Surgery_Status", "Chemotherapy", "Radiation", 
    "Marital_status_at_diagnosis", "Regional_Nodes_Involvement", "FIGO_Stage"
]
TIME_COL = "SurvivalMonths"
EVENT_COL = "Status"
TARGET_TIMES = [36, 48, 60] 
TIME_POINT = 60  # 【新增】5 年分层时间点

def load_data(path):
    df = pd.read_excel(path, na_values=["NaN", "nan", "NA", "Unknown"])
    df = df.dropna(subset=[TIME_COL, EVENT_COL])
    return df

print(">>> [Step 1] 加载数据...")
try:
    df_seer = load_data(path_seer)
    df_china = load_data(path_china)
except FileNotFoundError:
    print("❌ 错误：找不到文件，请检查路径是否正确。")
    raise 

# --- A. 内部数据 (SEER): 80% 训练 / 20% 验证 ---
X_seer = df_seer[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_seer = df_seer[[EVENT_COL, TIME_COL]]



# 计算 60 个月生存状态
survival_status_60m = get_60m_survival_status(y_seer, TIME_COL, EVENT_COL, TIME_POINT)

# 使用 60 个月生存状态进行分层 (替代原来的 stratify=y_seer[EVENT_COL])
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_seer, y_seer, 
    test_size=0.2, 
    random_state=42, 
    stratify=survival_status_60m  
)

# 验证分层效果
train_status = get_60m_survival_status(y_train_raw, TIME_COL, EVENT_COL, TIME_POINT)
val_status = get_60m_survival_status(y_val_raw, TIME_COL, EVENT_COL, TIME_POINT)

print(f"\n📊 60 个月生存状态分布对比:")
print(f"  训练集 (n={len(y_train_raw)}):")
print(f"    存活：{np.sum(train_status == 0)} ({np.sum(train_status == 0)/len(train_status)*100:.1f}%)")
print(f"    死亡：{np.sum(train_status == 1)} ({np.sum(train_status == 1)/len(train_status)*100:.1f}%)")

print(f"  验证集 (n={len(y_val_raw)}):")
print(f"    存活：{np.sum(val_status == 0)} ({np.sum(val_status == 0)/len(val_status)*100:.1f}%)")
print(f"    死亡：{np.sum(val_status == 1)} ({np.sum(val_status == 1)/len(val_status)*100:.1f}%)")

print(f"\n✅ 训练集/验证集 60 个月生存率差异：{abs(np.mean(train_status) - np.mean(val_status))*100:.2f}%")


X_ext_raw = df_china[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_ext_raw = df_china[[EVENT_COL, TIME_COL]]

print(f"\n数据概览：Internal Train={len(X_train_raw)}, Internal Val={len(X_val_raw)}, External Test={len(X_ext_raw)}")

print("\n>>> [Step 2] 预处理 (Fit on SEER Train ONLY)...")

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), NUMERIC_FEATURES),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_FEATURES)
], verbose_feature_names_out=False)

# 1. 只在 SEER 训练集上 Fit
preprocessor.fit(X_train_raw)



In [ ]:
# 2. 转换所有数据集
cols = preprocessor.get_feature_names_out()
X_train = pd.DataFrame(preprocessor.transform(X_train_raw), columns=cols).astype('float32')
X_val   = pd.DataFrame(preprocessor.transform(X_val_raw), columns=cols).astype('float32')
X_ext   = pd.DataFrame(preprocessor.transform(X_ext_raw), columns=cols).astype('float32') 

# 3. 处理 Y
def get_y(df): return Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL].values)
y_train = get_y(y_train_raw)
y_val   = get_y(y_val_raw)
y_ext   = get_y(y_ext_raw)

# ==========================================
# 3. Optuna + 5 折交叉验证
# ==========================================
print("\n>>> [Step 3] Optuna 参数优化 (5-Fold CV)...")

def get_cv_score(model_class, params, X, y, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X, y['event']):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        model = model_class(**params)
        try:
            model.fit(X_tr, y_tr)
            pred = model.predict(X_va)
            risk = -pred if isinstance(model, FastSurvivalSVM) else pred
            c = concordance_index_censored(y_va['event'], y_va['time'], risk)[0]
            scores.append(c)
        except: continue
    return np.mean(scores) if scores else 0.5

# --- RSF ---
def obj_rsf(trial):
    params = {'n_estimators': trial.suggest_int('n_estimators', 100, 300),
              'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 30), 
              'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
              'n_jobs': -1, 'random_state': 42}
    return get_cv_score(RandomSurvivalForest, params, X_train, y_train)

study_rsf = optuna.create_study(direction='maximize')
study_rsf.optimize(obj_rsf, n_trials=50)

# --- XGBoost ---
def obj_xgb(trial):
    params = {
        'objective': 'survival:cox',
        'tree_method': 'hist',
        'grow_policy': 'lossguide',

        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 1.0, log=True),
        

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5), 
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        'n_jobs': -1, 'random_state': 42, 'verbosity': 0
    }
    
    # 使用 CV 评分
    return get_cv_score(XGBoostSurvival, params, X_train, y_train, n_splits=5)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(obj_xgb, n_trials=50)

# --- SVM ---
def obj_svm(trial):
    params = {'alpha': trial.suggest_float('alpha', 0.01, 10.0, log=True),
              'rank_ratio': trial.suggest_float('rank_ratio', 0.0, 1.0),
              'max_iter': 1000, 'tol': 1e-4, 'random_state': 42}
    return get_cv_score(FastSurvivalSVM, params, X_train, y_train)

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(obj_svm, n_trials=50)

# ==========================================
# 【新增】输出最佳参数
# ==========================================
print("\n" + "="*70)
print("📋 Optuna 优化结果汇总")
print("="*70)
print(f"\n✅ RSF 最佳 C-index: {study_rsf.best_value:.4f}")
print(f"   最佳参数：{study_rsf.best_params}")

print(f"\n✅ XGBoost 最佳 C-index: {study_xgb.best_value:.4f}")
print(f"   最佳参数：{study_xgb.best_params}")

print(f"\n✅ SVM 最佳 C-index: {study_svm.best_value:.4f}")
print(f"   最佳参数：{study_svm.best_params}")

print("\n" + "="*70)
print("="*70)

In [ ]:
# ==========================================
# 4. 加载最佳参数并训练最终模型
# ==========================================
print("\n>>> [Step 4] 加载最佳参数并训练最终模型...")

import json
import pickle

# 加载最佳参数文件
with open('./trained_models/best_params.json', 'r', encoding='utf-8') as f:
    best_params = json.load(f)

print("✅ 最佳参数加载成功:")
for model_name, params in best_params.items():
    if model_name not in ['target_times', 'feature_names', 'n_features', 'n_train_samples']:
        print(f"   • {model_name}: {params}")

# 提取参数
RSF_PARAMS = best_params['RSF']
XGB_PARAMS = best_params['XGBoost']
SVM_PARAMS = best_params['SVM']
TARGET_TIMES = best_params['target_times']
FEATURE_NAMES = best_params['feature_names']

# ==========================================
# 4.1 训练最终模型 (使用全量 SEER Train)
# ==========================================
print("\n>>> [Step 4.1] 训练最终模型 (全量 SEER Train)...")

from sksurv.ensemble import RandomSurvivalForest
from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxPHSurvivalAnalysis

# RSF
print("   🌲 训练 RSF...")
best_rsf = RandomSurvivalForest(
    n_estimators=RSF_PARAMS['n_estimators'],
    min_samples_leaf=RSF_PARAMS['min_samples_leaf'],
    max_features=RSF_PARAMS['max_features'],
    n_jobs=-1,
    random_state=42,
    verbose=0
).fit(X_train, y_train)
print(f"      ✅ RSF 训练完成 (n_estimators={RSF_PARAMS['n_estimators']})")

# XGBoost
print("   📈 训练 XGBoost...")
best_xgb = XGBoostSurvival(
    n_estimators=XGB_PARAMS['n_estimators'],
    learning_rate=XGB_PARAMS['learning_rate'],
    max_depth=XGB_PARAMS['max_depth'],
    reg_alpha=XGB_PARAMS['reg_alpha'],
    random_state=42,
    verbose=0
).fit(X_train, y_train)
print(f"      ✅ XGBoost 训练完成 (n_estimators={XGB_PARAMS['n_estimators']})")

# SVM
print("   ⚡ 训练 SVM...")
best_svm = FastSurvivalSVM(
    alpha=SVM_PARAMS['alpha'],
    rank_ratio=SVM_PARAMS['rank_ratio'],
    max_iter=2000,
    random_state=42,
    verbose=0
).fit(X_train, y_train)
print(f"      ✅ SVM 训练完成 (alpha={SVM_PARAMS['alpha']:.4f})")

models_map = {"RSF": best_rsf, "XGBoost": best_xgb, "SVM": best_svm}

# ==========================================
# 5. 概率生成 (Cox Bridge - Direct Transfer)
# ==========================================
print("\n>>> [Step 5] 生成概率 (Cox Bridge - Direct Transfer)...")

def get_probs_direct(model, X_train_ref, y_train_ref, X_target, times):
    """
    Direct Transfer Logic:
    1. 用 SEER Train (X_train_ref) 训练 Cox Bridge (基线风险)。
    2. 用这个 SEER 的基线去预测 X_target。
    """
    # 1. 拿训练集分数
    tr_scores = model.predict(X_train_ref)
    if isinstance(model, FastSurvivalSVM): 
        tr_scores = -tr_scores  # SVM 输出负风险，需取反
        
    # 2. 拟合基线 (Fit on SEER Train)
    bridge = CoxPHSurvivalAnalysis()
    bridge.fit(pd.DataFrame({'score': tr_scores}), y_train_ref)
    
    # 3. 拿目标集分数
    target_scores = model.predict(X_target)
    if isinstance(model, FastSurvivalSVM): 
        target_scores = -target_scores
        
    # 4. 预测 (Apply SEER Baseline)
    surv_funcs = bridge.predict_survival_function(pd.DataFrame({'score': target_scores}))
    probs = np.array([[fn(t) for t in times] for fn in surv_funcs])
    
    return probs, target_scores, bridge

# 批量生成
results = []
for name, model in models_map.items():
    print(f"\n📊 处理 {name} 模型...")
    # Internal Val
    probs_int, scores_int, bridge_int = get_probs_direct(model, X_train, y_train, X_val, TARGET_TIMES)
    # External Test
    probs_ext, scores_ext, bridge_ext = get_probs_direct(model, X_train, y_train, X_ext, TARGET_TIMES)
    
    results.append({
        "Model": name,
        "Probs_Int": probs_int, 
        "Scores_Int": scores_int,
        "Probs_Ext": probs_ext, 
        "Scores_Ext": scores_ext,
        "Bridge": bridge_ext
    })
    print(f"   ✅ {name} 处理完成")

# ==========================================
# 6. 指标计算与展示
# ==========================================
print("\n>>> [Step 6] 计算全指标 (Internal vs External)...")

def find_best_f1(y_true, y_prob):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    numerator = 2 * precision * recall
    denominator = precision + recall
    f1 = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)
    return np.max(f1) if len(f1)>0 else 0.0

metrics_data = []

for res in results:
    model_name = res['Model']
    print(f"\n📈 计算 {model_name} 指标...")
    
    # --- Internal ---
    c_int = concordance_index_censored(y_val['event'], y_val['time'], res['Scores_Int'])[0]
    metrics_data.append({"Model": model_name, "Dataset": "Internal", "C-index": c_int})
    
    # --- External ---
    c_ext = concordance_index_censored(y_ext['event'], y_ext['time'], res['Scores_Ext'])[0]
    metrics_data.append({"Model": model_name, "Dataset": "External", "C-index": c_ext})
    
    # --- Time-dependent Metrics ---
    for i, t in enumerate(TARGET_TIMES):
        # Internal
        mask_int = (y_val['time'] > t) | (y_val['event'] == True)
        y_bin_int = (y_val['time'] <= t).astype(int)[mask_int]
        prob_death_int = 1 - res['Probs_Int'][mask_int, i]
        
        auc_int = roc_auc_score(y_bin_int, prob_death_int) if len(np.unique(y_bin_int))>1 else np.nan
        f1_int = find_best_f1(y_bin_int, prob_death_int)
        _, bs_int = brier_score(y_train, y_val, res['Probs_Int'][:, i], times=[t])
        
        idx_int = len(metrics_data) - 2
        metrics_data[idx_int][f"AUC {t}m"] = auc_int
        metrics_data[idx_int][f"F1 {t}m"] = f1_int
        metrics_data[idx_int][f"Brier {t}m"] = bs_int[0]
        
        # External
        mask_ext = (y_ext['time'] > t) | (y_ext['event'] == True)
        y_bin_ext = (y_ext['time'] <= t).astype(int)[mask_ext]
        prob_death_ext = 1 - res['Probs_Ext'][mask_ext, i]
        
        auc_ext = roc_auc_score(y_bin_ext, prob_death_ext) if len(np.unique(y_bin_ext))>1 else np.nan
        f1_ext = find_best_f1(y_bin_ext, prob_death_ext)
        _, bs_ext = brier_score(y_ext, y_ext, res['Probs_Ext'][:, i], times=[t])
        
        idx_ext = len(metrics_data) - 1
        metrics_data[idx_ext][f"AUC {t}m"] = auc_ext
        metrics_data[idx_ext][f"F1 {t}m"] = f1_ext
        metrics_data[idx_ext][f"Brier {t}m"] = bs_ext[0]
    
    print(f"   ✅ {model_name} 指标计算完成")

df_metrics = pd.DataFrame(metrics_data).set_index(["Model", "Dataset"])
cols = ["C-index"] + [f"{m} {t}m" for t in TARGET_TIMES for m in ["AUC", "Brier", "F1"]]
df_metrics = df_metrics[cols]

print("\n" + "="*80)
print(df_metrics.applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x))
print("="*80)

# ==========================================
# 7. 绘图 (ROC)
# ==========================================
print("\n>>> [Step 7] 绘制 ROC 曲线...")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col_idx, t in enumerate(TARGET_TIMES):
    # Internal
    ax1 = axes[0, col_idx]
    for res in results:
        mask = (y_val['time'] > t) | (y_val['event'] == True)
        y_bin = (y_val['time'] <= t).astype(int)[mask]
        scores = res['Scores_Int'][mask]
        fpr, tpr, _ = roc_curve(y_bin, scores)
        auc_val = auc(fpr, tpr)
        ax1.plot(fpr, tpr, lw=2, label=f"{res['Model']} (AUC={auc_val:.3f})")
    ax1.plot([0,1],[0,1],'k--')
    ax1.set_title(f"Internal ROC ({t}m)")
    ax1.legend()
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    
    # External
    ax2 = axes[1, col_idx]
    for res in results:
        mask = (y_ext['time'] > t) | (y_ext['event'] == True)
        y_bin = (y_ext['time'] <= t).astype(int)[mask]
        scores = res['Scores_Ext'][mask]
        fpr, tpr, _ = roc_curve(y_bin, scores)
        auc_val = auc(fpr, tpr)
        ax2.plot(fpr, tpr, lw=2, label=f"{res['Model']} (AUC={auc_val:.3f})")
    ax2.plot([0,1],[0,1],'k--')
    ax2.set_title(f"External ROC ({t}m)")
    ax2.legend()
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")

plt.tight_layout()
plt.savefig('./model_comparison_roc0228.png', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# ==========================================

# ==========================================
print("\n>>> [Step 6-Extended] 计算 95% CI + Sensitivity/Specificity...")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, precision_recall_curve, brier_score_loss
from sksurv.metrics import concordance_index_censored
import warnings
warnings.filterwarnings('ignore')

# ========== 配置参数 ==========
TARGET_TIMES = [36, 48, 60]
N_BOOTSTRAP = 500
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ========== 辅助函数 ==========

def find_youden_threshold(y_true, y_prob):
    """使用 Youden 指数找到最佳阈值"""
    if len(np.unique(y_true)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    optimal_idx = np.argmax(youden)
    return thresholds[optimal_idx]

def calc_sen_spec_f1(y_true, y_prob):
    """基于 Youden 阈值计算 Sensitivity/Specificity/F1"""
    if len(np.unique(y_true)) < 2 or len(y_true) < 10:
        return np.nan, np.nan, np.nan
    
    thresh = find_youden_threshold(y_true, y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else np.nan
    
    return sensitivity, specificity, f1

def bootstrap_ci_2d(y_true, y_prob, stat_func, n_boot=500, ci=0.95):
    """Bootstrap CI 计算 (需要 y_true + y_prob 的统计量)"""
    if len(y_true) < 20 or len(np.unique(y_true)) < 2:
        return np.nan, (np.nan, np.nan)
    
    orig_val = stat_func(y_true, y_prob)
    boot_vals = []
    
    for _ in range(n_boot):
        idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            boot_vals.append(stat_func(y_true[idx], y_prob[idx]))
        except:
            continue
    
    if len(boot_vals) < 50:
        return orig_val, (np.nan, np.nan)
    
    return orig_val, (np.percentile(boot_vals, (1-ci)/2*100), np.percentile(boot_vals, (1+ci)/2*100))

def bootstrap_ci_sen_spec_f1(y_true, y_prob, n_boot=500, ci=0.95):
    """Bootstrap CI 计算 Sensitivity/Specificity/F1"""
    orig_sen, orig_spec, orig_f1 = calc_sen_spec_f1(y_true, y_prob)
    
    if np.isnan(orig_sen):
        return (np.nan, np.nan, np.nan), ((np.nan, np.nan), (np.nan, np.nan), (np.nan, np.nan))
    
    boot_sen, boot_spec, boot_f1 = [], [], []
    
    for _ in range(n_boot):
        idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        sen, spec, f1 = calc_sen_spec_f1(y_true[idx], y_prob[idx])
        if not np.isnan(sen):
            boot_sen.append(sen)
            boot_spec.append(spec)
            boot_f1.append(f1)
    
    def get_ci(vals):
        if len(vals) < 50:
            return (np.nan, np.nan)
        return (np.percentile(vals, (1-ci)/2*100), np.percentile(vals, (1+ci)/2*100))
    
    return (orig_sen, orig_spec, orig_f1), (get_ci(boot_sen), get_ci(boot_spec), get_ci(boot_f1))

def bootstrap_c_index_ci(event, time, risk_score, n_boot=500, ci=0.95):
    """C-index Bootstrap CI"""
    c_orig = concordance_index_censored(event.astype(bool), time, risk_score)[0]
    boot_c = []
    n = len(event)
    
    for _ in range(n_boot):
        idx = np.random.choice(n, size=n, replace=True)
        if np.sum(event[idx].astype(bool)) < 2:
            continue
        try:
            c_boot = concordance_index_censored(event[idx].astype(bool), time[idx], risk_score[idx])[0]
            boot_c.append(c_boot)
        except:
            continue
    
    if len(boot_c) < 50:
        return c_orig, (np.nan, np.nan)
    
    return c_orig, (np.percentile(boot_c, (1-ci)/2*100), np.percentile(boot_c, (1+ci)/2*100))

def calculate_brier_ci_simple(y_true, y_prob, n_boot=500, ci=0.95):
    """
    【修复】简化版 Brier Score + Bootstrap CI
    使用标准二分类 Brier Score 公式，避免 sksurv 的复杂依赖
    """
    if len(y_true) < 20 or len(np.unique(y_true)) < 2:
        return np.nan, (np.nan, np.nan)
    
    # 原始 Brier Score
    bs_orig = np.mean((y_prob - y_true) ** 2)
    
    # Bootstrap CI
    boot_bs = []
    for _ in range(n_boot):
        idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        bs_b = np.mean((y_prob[idx] - y_true[idx]) ** 2)
        boot_bs.append(bs_b)
    
    if len(boot_bs) < 50:
        return bs_orig, (np.nan, np.nan)
    
    ci_lower = np.percentile(boot_bs, (1-ci)/2 * 100)
    ci_upper = np.percentile(boot_bs, (1+ci)/2 * 100)
    
    return bs_orig, (ci_lower, ci_upper)

def format_with_ci(value, ci_lower, ci_upper):
    """格式化输出：Value (CI_Lower, CI_Upper)"""
    if np.isnan(value) or np.isnan(ci_lower):
        return "N/A"
    return f"{value:.3f} ({ci_lower:.3f}, {ci_upper:.3f})"

# ========== 主计算函数 ==========

print("\n📊 开始计算各指标 + 95% CI...")

metrics_data = []

for res in results:
    model_name = res['Model']
    print(f"\n🔹 模型：{model_name}")
    
    # 动态检测键名
    if 'Probs_Int' in res:
        probs_int_key, scores_int_key = 'Probs_Int', 'Scores_Int'
    elif 'Probs_Val' in res:
        probs_int_key, scores_int_key = 'Probs_Val', 'Scores_Val'
    else:
        raise KeyError(f"找不到概率键名！results 中的键：{list(res.keys())}")
    
    for dataset_name, y_data, probs, scores in [
        ('Internal', y_val, res[probs_int_key], res[scores_int_key]),
        ('External', y_ext, res['Probs_Ext'], res['Scores_Ext'])
    ]:
        print(f"   • {dataset_name} Validation...")
        
        # 【关键】第 1 列=Model，第 2 列=Dataset
        row = {
            'Model': model_name,
            'Dataset': dataset_name
        }
        
        # 1. C-index + CI
        c_val, (c_l, c_u) = bootstrap_c_index_ci(
            y_data['event'], y_data['time'], scores, n_boot=N_BOOTSTRAP
        )
        row['C-index'] = format_with_ci(c_val, c_l, c_u)
        
        # 2. 时间依赖性指标
        for i, t in enumerate(TARGET_TIMES):
            mask = (y_data['time'] > t) | (y_data['event'] == True)
            if np.sum(mask) < 20:
                for m in ['AUC', 'Brier', 'F1', 'Sensitivity', 'Specificity']:
                    row[f'{t}m_{m}'] = "N/A"
                continue
            
            y_bin = ((y_data['time'] <= t) & (y_data['event'] == True)).astype(int)[mask]
            prob_death = 1 - probs[mask, i]
            
            # AUC + CI
            if len(np.unique(y_bin)) > 1:
                auc_val, (auc_l, auc_u) = bootstrap_ci_2d(
                    y_bin, prob_death, roc_auc_score, n_boot=N_BOOTSTRAP
                )
                row[f'{t}m_AUC'] = format_with_ci(auc_val, auc_l, auc_u)
            else:
                row[f'{t}m_AUC'] = "N/A"
            
            # Brier Score + CI【修复】使用简化版计算
            bs_val, (bs_l, bs_u) = calculate_brier_ci_simple(
                y_bin, prob_death, n_boot=N_BOOTSTRAP
            )
            row[f'{t}m_Brier'] = format_with_ci(bs_val, bs_l, bs_u)
            
            # Sensitivity/Specificity/F1 + CI
            (sen, spec, f1), ((sen_l, sen_u), (spec_l, spec_u), (f1_l, f1_u)) = bootstrap_ci_sen_spec_f1(
                y_bin, prob_death, n_boot=N_BOOTSTRAP
            )
            
            row[f'{t}m_Sensitivity'] = format_with_ci(sen, sen_l, sen_u)
            row[f'{t}m_Specificity'] = format_with_ci(spec, spec_l, spec_u)
            row[f'{t}m_F1'] = format_with_ci(f1, f1_l, f1_u)
        
        metrics_data.append(row)

# ========== 整理为论文级表格 ==========

df_metrics = pd.DataFrame(metrics_data)

# 【关键】强制列顺序：Model → Dataset → C-index → 各时间点指标
base_cols = ['Model', 'Dataset', 'C-index']
time_cols = []
for t in TARGET_TIMES:
    for m in ['AUC', 'Brier', 'F1', 'Sensitivity', 'Specificity']:
        time_cols.append(f'{t}m_{m}')

df_metrics = df_metrics[base_cols + time_cols]

# ========== 输出预览 ==========

print("\n" + "="*150)
print("="*150)

for dataset in ['Internal', 'External']:
    print(f"\n🔹 {dataset} Validation:")
    subset = df_metrics[df_metrics['Dataset'] == dataset]
    display_cols = ['Model', 'Dataset', 'C-index'] + [c for c in time_cols if '60m' in c]
    print(subset[display_cols].to_string(index=False))

# ========== 导出 Excel ==========


In [ ]:
# ==========================================
# 10. 风险分层：RSF 三分类 (内外部独立阈值) - 左右双子图
# ==========================================
print("\n>>> [Step 10] 风险分层：RSF 三分类 (Internal & External, Independent Thresholds)...")

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from sklearn.metrics import roc_curve

def find_youden_threshold(y_true, y_prob):
    """使用 Youden 指数找到最佳分类阈值"""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    return thresholds[optimal_idx], youden_index[optimal_idx], tpr[optimal_idx], 1 - fpr[optimal_idx]

def stratify_3class(prob_death, thresh_youden, thresh_25):
    """三分类：0=Low, 1=Medium, 2=High"""
    risk_groups = np.zeros(len(prob_death), dtype=int)
    is_high = prob_death > thresh_youden
    risk_groups[is_high] = 2
    risk_groups[~is_high] = np.where(prob_death[~is_high] > thresh_25, 1, 0)
    return risk_groups

# 仅处理 RSF 模型
rsf_result = [res for res in results if res['Model'] == 'RSF'][0]
model_name = 'RSF'
t_point = 60
col_idx = 2  # TARGET_TIMES[2] = 60

print(f"\n📊 {model_name} 三分类风险分层计算 (独立阈值)...")

# ==========================================
# Step 1: 分别计算内部 + 外部验证集的阈值
# ==========================================

# --- Internal Validation ---
print("\n🔹 Internal Validation:")
mask_det_val = (y_val['time'] > t_point) | (y_val['event'] == True)
y_bin_val = ((y_val['time'] <= t_point) & (y_val['event'] == True)).astype(int)[mask_det_val]
prob_death_val_det = 1 - rsf_result['Probs_Int'][mask_det_val, col_idx]

thresh_youden_val, youden_val, _, _ = find_youden_threshold(y_bin_val, prob_death_val_det)
prob_death_val_all = 1 - rsf_result['Probs_Int'][:, col_idx]
prob_non_high_val = prob_death_val_all[prob_death_val_all <= thresh_youden_val]
thresh_25_val = np.percentile(prob_non_high_val, 25) if len(prob_non_high_val) > 0 else 0

print(f"   ✓ Youden 阈值：{thresh_youden_val:.3f}")
print(f"   ✓ 25% 分位数：{thresh_25_val:.3f}")

risk_groups_val = stratify_3class(prob_death_val_all, thresh_youden_val, thresh_25_val)

# --- External Validation ---
print("\n🔹 External Validation:")
mask_det_ext = (y_ext['time'] > t_point) | (y_ext['event'] == True)
y_bin_ext = ((y_ext['time'] <= t_point) & (y_ext['event'] == True)).astype(int)[mask_det_ext]
prob_death_ext_det = 1 - rsf_result['Probs_Ext'][mask_det_ext, col_idx]

thresh_youden_ext, youden_ext, _, _ = find_youden_threshold(y_bin_ext, prob_death_ext_det)
prob_death_ext_all = 1 - rsf_result['Probs_Ext'][:, col_idx]
prob_non_high_ext = prob_death_ext_all[prob_death_ext_all <= thresh_youden_ext]
thresh_25_ext = np.percentile(prob_non_high_ext, 25) if len(prob_non_high_ext) > 0 else 0

print(f"   ✓ Youden 阈值：{thresh_youden_ext:.3f}")
print(f"   ✓ 25% 分位数：{thresh_25_ext:.3f}")

risk_groups_ext = stratify_3class(prob_death_ext_all, thresh_youden_ext, thresh_25_ext)

# 统计
def count_groups(risk_groups):
    n = len(risk_groups)
    return {
        'Low': np.sum(risk_groups == 0),
        'Med': np.sum(risk_groups == 1), 
        'High': np.sum(risk_groups == 2),
        'Total': n
    }

stats_val = count_groups(risk_groups_val)
stats_ext = count_groups(risk_groups_ext)

print(f"\n📊 分组统计:")
print(f"   • Internal (n={stats_val['Total']}): Low={stats_val['Low']}, Med={stats_val['Med']}, High={stats_val['High']}")
print(f"   • External (n={stats_ext['Total']}): Low={stats_ext['Low']}, Med={stats_ext['Med']}, High={stats_ext['High']}")

# ==========================================
# Step 2: 绘制左右双子图 KM 曲线
# ==========================================
print("\n>>> [Step 11] 绘制左右双子图 KM 曲线...")

# 全局字体设置
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10

# 创建大图：1 行 2 列 (左右并排)
fig_km, (ax_val, ax_ext) = plt.subplots(1, 2, figsize=(16, 7))
fig_km.suptitle('Kaplan-Meier Survival Curves by Risk Stratification (RSF Model - 3 Groups)', 
                fontsize=15, fontweight='bold', y=1.02, fontname='Arial')

# 统一样式
colors_3risk = {0: '#2E86AB', 1: '#F39C12', 2: '#E74C3C'}  # Blue, Orange, Red
risk_names_3 = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
line_styles_3 = {0: '-', 1: '--', 2: '-.'}

# 绘制函数
def plot_km_subplot(ax, y_surv, risk_groups, thresh_youden, thresh_25, dataset_label):
    """绘制单个 KM 子图"""
    
    for risk_level in [0, 1, 2]:
        mask_risk = (risk_groups == risk_level)
        if np.sum(mask_risk) < 10:
            continue
        
        kmf = KaplanMeierFitter()
        kmf.fit(durations=y_surv['time'][mask_risk], 
                event_observed=y_surv['event'][mask_risk],
                label=f"{risk_names_3[risk_level]} (n={np.sum(mask_risk)})")
        
        # 仅 High Risk 显示置信区间，避免杂乱
        kmf.plot_survival_function(ax=ax, ci_show=(risk_level==2), 
                                   color=colors_3risk[risk_level],
                                   linestyle=line_styles_3[risk_level],
                                   linewidth=2.5)
    
    # Log-rank 检验 (Low vs High)
    mask_low = (risk_groups == 0)
    mask_high = (risk_groups == 2)
    p_text = "P = N/A"
    
    if np.sum(mask_low) > 5 and np.sum(mask_high) > 5:
        result = logrank_test(y_surv['time'][mask_low], y_surv['time'][mask_high],
                             event_observed_A=y_surv['event'][mask_low],
                             event_observed_B=y_surv['event'][mask_high])
        p_val = result.p_value
        p_text = f"P = {p_val:.10f}"
        print(p_text)
    
    # 子图标题 (显示数据集名称 + 各自阈值)
    title_str = f'{dataset_label}\nYouden={thresh_youden:.3f}, 25%={thresh_25:.3f}'
    ax.set_title(title_str, fontweight='bold', fontname='Arial', fontsize=11, pad=12)
    
    ax.set_xlabel('Time (months)', fontname='Arial', fontsize=11)
    ax.set_ylabel('Survival Probability', fontname='Arial', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.set_ylim([0, 1.05])
    ax.set_xlim([0, 72])
    p_text = "p<0.0001"
    # 【修改】P 值显示在子图左下角 (无图例)
    ax.text(0.02, 0.05,p_text , transform=ax.transAxes, fontsize=16, 
            fontname='Arial', ha='left', va='bottom')
    
    # 【修改】移除子图内图例：不调用 ax.legend()
    
    # 统一字体
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname('Arial')
        label.set_fontsize(10)
    
    return ax

# 绘制 Left: Internal Validation
plot_km_subplot(ax_val, y_val, risk_groups_val, 
                thresh_youden_val, thresh_25_val, 
                dataset_label='Internal Validation')

# 绘制 Right: External Validation
plot_km_subplot(ax_ext, y_ext, risk_groups_ext, 
                thresh_youden_ext, thresh_25_ext, 
                dataset_label='External Validation')

# 【关键】统一图例放在大图底部中央 (仅显示风险组，不显示 P 值)
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=colors_3risk[0], linestyle=line_styles_3[0], lw=2.5, label='Low Risk'),
    Line2D([0], [0], color=colors_3risk[1], linestyle=line_styles_3[1], lw=2.5, label='Medium Risk'),
    Line2D([0], [0], color=colors_3risk[2], linestyle=line_styles_3[2], lw=2.5, label='High Risk'),
    Line2D([0], [0], color=colors_3risk[2], linestyle='-', lw=1, alpha=0.5, label='95% CI (High)')
]

fig_km.legend(handles=legend_elements, labels=[e.get_label() for e in legend_elements],
              loc='lower center', ncol=4, frameon=True, fontsize=11, 
              bbox_to_anchor=(0.5, -0.03), borderaxespad=0, framealpha=0.9)

# 调整布局，为底部图例留出空间
plt.tight_layout(rect=[0, 0.05, 1, 0.96])

# 保存并显示
plt.savefig('km_stratification_rsf_internal_external_3class.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("✅ KM 双子图已保存：km_stratification_rsf_internal_external_3class.png")


In [ ]:
# ==========================================
# DCA 净获益表格：所有阈值 + NB>0 区间高亮
# ==========================================
print("\n>>> [Step] 生成 DCA 净获益表格 (所有阈值 + NB>0 高亮)...")

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ========== 配置参数 ==========
TARGET_TIMES = [36, 48, 60]
THRESHOLDS = np.round(np.linspace(0.01, 1, 100), 2)  # 60 个阈值: 0.01~0.60

# ========== 辅助函数 ==========

def prepare_binary_data(y_surv, probs, time_point, col_idx):
    """准备二分类数据"""
    mask = (y_surv['time'] > time_point) | (y_surv['event'] == True)
    if np.sum(mask) == 0:
        return None, None
    y_bin = ((y_surv['time'] <= time_point) & (y_surv['event'] == True)).astype(int)[mask]
    prob_death = 1 - probs[mask, col_idx]
    return y_bin, prob_death

def calculate_net_benefit(y_true, y_prob, thresholds):
    """计算净获益"""
    net_benefits = []
    n = len(y_true)
    if n == 0:
        return np.zeros_like(thresholds)
    
    for thresh in thresholds:
        pred_pos = (y_prob >= thresh)
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        if thresh < 1.0:
            nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        else:
            nb = tp / n
        net_benefits.append(nb)
    return np.array(net_benefits)

# ========== 主计算 ==========

print("\n📊 计算所有阈值的净获益...")

# 存储结果：{ (model, dataset, time): {threshold: nb} }
nb_results = {}

for res in results:
    model = res['Model']
    
    # 动态检测键名
    probs_key = 'Probs_Int' if 'Probs_Int' in res else ('Probs_Val' if 'Probs_Val' in res else None)
    if probs_key is None:
        print(f"   ⚠️  跳过 {model}: 找不到概率数据")
        continue
    
    for dataset_name, y_data, probs in [
        ('Internal', y_val, res[probs_key]),
        ('External', y_ext, res['Probs_Ext'])
    ]:
        for i, t in enumerate(TARGET_TIMES):
            y_bin, prob_death = prepare_binary_data(y_data, probs, t, i)
            if y_bin is None or len(y_bin) < 20:
                continue
            
            nb_values = calculate_net_benefit(y_bin, prob_death, THRESHOLDS)
            key = (model, dataset_name, f'{t}m')
            nb_results[key] = dict(zip(THRESHOLDS, np.round(nb_values, 4)))
            print(f"   ✅ {model} - {dataset_name} - {t}m: 计算完成")

# ========== 生成表格 ==========

print("\n📋 生成宽格式表格 (阈值×模型×数据集×时间点)...")

# 方法 1: 长格式 (方便筛选)
long_data = []
for (model, dataset, time_point), nb_dict in nb_results.items():
    for thresh, nb in nb_dict.items():
        long_data.append({
            'Threshold': thresh,
            'Model': model,
            'Dataset': dataset,
            'Time_Point': time_point,
            'Net_Benefit': nb,
            'NB_Positive': '✅ YES' if nb > 0 else '❌ NO'  # 标记 NB>0
        })

df_long = pd.DataFrame(long_data)

# 方法 2: 宽格式 (阈值作为行，方便横向对比)
print("\n📋 生成宽格式表格 (阈值作为行)...")

wide_data = {'Threshold': THRESHOLDS}
for (model, dataset, time_point), nb_dict in nb_results.items():
    col_name = f"{model}_{dataset}_{time_point}"
    wide_data[col_name] = [nb_dict.get(t, np.nan) for t in THRESHOLDS]

df_wide = pd.DataFrame(wide_data)

# 方法 3: 透视表格式 (Model×Dataset 为列)
print("\n📋 生成透视表格式...")

df_pivot = df_long.pivot_table(
    index='Threshold',
    columns=['Model', 'Dataset', 'Time_Point'],
    values='Net_Benefit',
    aggfunc='first'
).round(4)

# ========== 输出关键信息 ==========

print("\n" + "="*100)
print("📊 NB > 0 的阈值区间汇总")
print("="*100)

for (model, dataset, time_point), nb_dict in nb_results.items():
    # 找出 NB > 0 的连续区间
    positive_thresh = [t for t, nb in nb_dict.items() if nb > 0]
    
    if len(positive_thresh) == 0:
        print(f"\n❌ {model} - {dataset} - {time_point}: 无 NB > 0 的区间")
        continue
    
    # 找出连续区间
    intervals = []
    if positive_thresh:
        start = positive_thresh[0]
        end = positive_thresh[0]
        for i in range(1, len(positive_thresh)):
            if abs(positive_thresh[i] - end) < 0.02:  # 连续阈值
                end = positive_thresh[i]
            else:
                intervals.append((start, end))
                start = end = positive_thresh[i]
        intervals.append((start, end))
    
    max_nb = max(nb_dict.values())
    max_nb_thresh = [t for t, nb in nb_dict.items() if nb == max_nb][0]
    
    print(f"\n✅ {model} - {dataset} - {time_point}:")
    print(f"   • NB > 0 区间: {', '.join([f'[{i[0]:.2f}, {i[1]:.2f}]' for i in intervals])}")
    print(f"   • 最大 NB: {max_nb:.4f} @ Threshold = {max_nb_thresh:.2f}")
    print(f"   • 阈值覆盖: {len(positive_thresh)}/{len(THRESHOLDS)} ({100*len(positive_thresh)/len(THRESHOLDS):.1f}%)")

# ========== 导出 Excel (带条件格式) ==========

print("\n>>> 导出 Excel (带 NB>0 高亮)...")

with pd.ExcelWriter('dca_net_benefit_all_thresholds.xlsx', engine='openpyxl') as writer:
    
    # Sheet 1: 长格式 (方便筛选)
    df_long.to_excel(writer, sheet_name='Long_Format', index=False)
    
    # Sheet 2: 宽格式 (阈值×各模型)
    df_wide.to_excel(writer, sheet_name='Wide_Format', index=False)
    
    # Sheet 3: 透视表
    df_pivot.to_excel(writer, sheet_name='Pivot_Format')
    
    # Sheet 4: NB>0 区间汇总 (简洁版)
    summary_data = []
    for (model, dataset, time_point), nb_dict in nb_results.items():
        positive_thresh = [t for t, nb in nb_dict.items() if nb > 0]
        if positive_thresh:
            intervals = []
            start = end = positive_thresh[0]
            for i in range(1, len(positive_thresh)):
                if abs(positive_thresh[i] - end) < 0.02:
                    end = positive_thresh[i]
                else:
                    intervals.append(f"{start:.2f}-{end:.2f}")
                    start = end = positive_thresh[i]
            intervals.append(f"{start:.2f}-{end:.2f}")
            max_nb = max(nb_dict.values())
            summary_data.append({
                'Model': model,
                'Dataset': dataset,
                'Time_Point': time_point,
                'NB_Positive_Range': ', '.join(intervals),
                'Max_NB': round(max_nb, 4),
                'Threshold_Coverage': f"{len(positive_thresh)}/{len(THRESHOLDS)}"
            })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_excel(writer, sheet_name='NB_Positive_Summary', index=False)

print("✅ Excel 已保存：dca_net_benefit_all_thresholds.xlsx")
print("   📁 Sheet 1: Long_Format (Threshold×Model×Dataset×Time×NB)")
print("   📁 Sheet 2: Wide_Format (阈值作为行，方便横向对比)")
print("   📁 Sheet 3: Pivot_Format (透视表)")
print("   📁 Sheet 4: NB_Positive_Summary (NB>0 区间汇总)")

# ========== 控制台预览 (前 20 行) ==========

print("\n" + "="*100)
print("📋 宽格式表格预览 (前 20 行，NB>0 用 ✅ 标记)")
print("="*100)

# 添加 NB>0 标记列
df_wide_marked = df_wide.copy()
for col in df_wide_marked.columns:
    if col != 'Threshold':
        df_wide_marked[col] = df_wide_marked[col].apply(lambda x: f"✅{x:.4f}" if pd.notna(x) and x > 0 else f"{x:.4f}" if pd.notna(x) else "N/A")

print(df_wide_marked.head(20).to_string(index=False))


In [ ]:
# ==========================================
# SHAP 个体力图分析 - 独立完整程序 (修复版)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🔬 SHAP 个体力图分析：三风险组典型病例展示")
print("="*80)

# ==========================================
# Step 1: 加载必要数据
# ==========================================
print("\n>>> [Step 1] 加载数据...")

try:
    assert best_rsf is not None, "best_rsf 模型未定义"
    assert X_ext is not None, "X_ext 未定义"
    assert y_ext is not None, "y_ext 未定义"
    assert risk_groups_ext is not None, "risk_groups_ext 未定义"
    assert prob_death_ext_all is not None, "prob_death_ext_all 未定义"
    print("   ✅ 所有必要数据已加载")
except Exception as e:
    print(f"   ⚠️  数据未加载：{e}")
    raise
=========================================

# ==========================================
# Step 3: 计算 SHAP 值
# ==========================================
print("\n>>> [Step 3] 计算 SHAP 值...")

X_ext_np = X_ext.values
n_features = X_ext_np.shape[1]


print("   • 初始化 SHAP KernelExplainer...")

def model_predict_risk(X):
    """模型预测风险评分"""
    X_df = pd.DataFrame(X, columns=X_ext.columns)
    pred = best_rsf.predict(X_df)
    return pred

background_indices = np.random.choice(len(X_ext_np), size=50, replace=False)
background_data = X_ext_np[background_indices]

explainer = shap.KernelExplainer(model_predict_risk, background_data)
print(f"   • 背景样本数：{len(background_data)}")

# 计算选中患者的 SHAP 值
shap_values_list = []
patient_info = []

for idx in all_sample_indices:
    print(f"   • 计算患者 {idx} 的 SHAP 值...")
    x_patient = X_ext_np[idx:idx+1]
    shap_val = explainer.shap_values(x_patient)
    shap_values_list.append(shap_val)
    
    info = {
        'Index': idx,
        'Risk_Group': ['Low', 'Medium', 'High'][risk_groups_ext[idx]],
        'Risk_Code': risk_groups_ext[idx],
        'FIGO_Stage': figo_ext[idx],
        'Survival_Months': float(y_ext['time'][idx]),
        'Event': int(y_ext['event'][idx]),
        'Pred_Death_Prob': float(prob_death_ext_all[idx]),
        'Pred_Survival_Prob': float(1 - prob_death_ext_all[idx])
    }
    patient_info.append(info)

df_patient_info = pd.DataFrame(patient_info)
print(f"\n📋 选中患者信息:")
print(df_patient_info[['Index', 'Risk_Group', 'FIGO_Stage', 'Event', 'Pred_Survival_Prob']].to_string(index=False))

# ==========================================
# Step 4: 绘制 SHAP force_plot (每个患者独立图像)
# ==========================================
print("\n>>> [Step 4] 绘制 SHAP force_plot...")

# 确保特征名数量与数据维度一致
feature_names_display = list(X_ext.columns)

# 风险组颜色
risk_color = {'Low': '#2E86AB', 'Medium': '#F39C12', 'High': '#E74C3C'}

# 【修复】为每个患者创建独立的 force_plot，然后组合成大图
fig_list = []

for i, idx in enumerate(all_sample_indices):
    info = patient_info[i]
    shap_val = shap_values_list[i][0]  # shape: (n_features,)
    
    # 获取患者原始特征值
    x_patient = X_ext_np[idx:idx+1]
    x_patient_df = pd.DataFrame(x_patient, columns=feature_names_display)
    
    print(f"   • 患者 {idx}: X shape={x_patient_df.shape}, SHAP shape={shap_val.shape}")
    
    # 【修复】创建独立图像，不使用 ax 参数
    fig = shap.force_plot(
        explainer.expected_value,
        shap_val,
        x_patient_df,
        feature_names=feature_names_display,
        matplotlib=True,
        show=False
    )
    
    # 添加患者信息标题
    fig.suptitle(
        f'Patient #{info["Index"]} | {info["Risk_Group"]} Risk | '
        f'FIGO {info["FIGO_Stage"]} | Event={info["Event"]} | '
        f'Pred Survival={info["Pred_Survival_Prob"]*100:.1f}%',
        fontsize=11, fontweight='bold', fontname='Arial', y=1.02,
        backgroundcolor=risk_color.get(info['Risk_Group'], 'white'))
    
    fig_list.append(fig)

# 将所有 force_plot 组合成一个大图
n_patients = len(fig_list)
n_cols = 2
n_rows = (n_patients + 1) // 2

fig_combined, axes = plt.subplots(n_rows, n_cols, figsize=(20, 8 * n_rows))

# 处理 axes 数组形状
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)

fig_combined.suptitle('SHAP Force Plots: Individual Risk Factor Contribution (RSF Model)', 
                      fontsize=15, fontweight='bold', y=0.995, fontname='Arial')



In [ ]:
# ==========================================
# Step 5: 绘制 SHAP force_plot (SHAP 值直接嵌入特征名)
# ==========================================
print("\n>>> [Step 5] 绘制 SHAP force_plot (SHAP 值嵌入特征名)...")

risk_color = {'Low': '#2E86AB', 'Medium': '#F39C12', 'High': '#E74C3C'}

for i, idx in enumerate(all_sample_indices):
    info = patient_info[i]
    display_data = display_data_list[i]
    
    # ========== 1. 处理 NaN 值 + 数值格式化 + 嵌入 SHAP 值 ==========
    final_names = []  # 最终用于显示的列表 (包含 SHAP 值)
    final_shap = []
    final_vals = []
    
    for j, (name, shap_v, val) in enumerate(zip(
        display_data['feature_names'], 
        display_data['shap_values'], 
        display_data['x_values']
    )):
        # --- A. 获取原始值 (用于 NaN 判断) ---
        if name == 'Age':
            original = df_china['Age'].iloc[idx]
        elif 'Tumor' in name or 'Size' in name:
            original = df_china['Tumor_Size'].iloc[idx]
        else:
            original = None
        
        # --- B. 处理 NaN ---
        if original is not None and pd.isna(original):
            display_val = val
        else:
            display_val = val
        
        # --- C. 数值格式化 (准备基础名称) ---
        if name == 'Age':
            display_val = int(round(display_val))
            base_name = 'Age'
        elif 'Tumor' in name or 'Size' in name:
            display_val_formatted = round(float(display_val), 2)
            base_name = 'Tumor_Size'
        elif '(' in name:
            # 分类变量已有括号，保持原样
            base_name = name
        else:
            # 其他二分类变量
            if isinstance(display_val, (int, float)) and display_val in [0, 1]:
                base_name = f'{name} ({int(display_val)})'
            else:
                base_name = name
        
        # --- D. 【核心修改】将 SHAP 值直接拼接到名称中 ---
        sign = '+' if shap_v > 0 else ''
        # 格式：特征名 (数值) [SHAP 值]
        # 例如：'Age (58y) [+0.156]' 或 'Surgery (Yes) [-0.045]'
        final_name = f"{base_name} [{sign}{shap_v:.3f}]"
        
        final_names.append(final_name)
        final_shap.append(shap_v)
        final_vals.append(display_val)
    
    # 创建显示用 DataFrame (使用包含 SHAP 值的最终名称)
    x_display_df = pd.DataFrame([final_vals], columns=final_names)
    
    # ========== 2. 筛选 Top 6 特征 ==========
    shap_vals_arr = np.array(final_shap)
    top_6_indices = np.argsort(np.abs(shap_vals_arr))[::-1][:4]
    
    top_6_names = [final_names[j] for j in top_6_indices]
    top_6_shap = shap_vals_arr[top_6_indices]
    top_6_vals = [final_vals[j] for j in top_6_indices]
    
    x_display_df_top6 = pd.DataFrame([top_6_vals], columns=top_6_names)
    
    print(f"   • 患者 {idx}: 显示 {len(top_6_names)} 个特征 (Top 6)")
    print(f"      示例特征名：{top_6_names[0]}")  # 调试：确认名称已包含 SHAP 值
    
    # ========== 3. 绘制 force_plot (自动显示嵌入的 SHAP 值) ==========
    fig = shap.force_plot(
        explainer.expected_value,
        top_6_shap,
        x_display_df_top6,
        feature_names=top_6_names,  # 传入包含 SHAP 值的名称
        matplotlib=True,
        show=False
    )
    
    # 去掉网格
    ax = fig.axes[0]
    ax.grid(False)
    
    # 【重要】不再需要手动添加 ax.text 标注，因为名称里已经包含了！
    
    
    # 保存
    fig.savefig(f'shap_force_plot_patient_{idx}.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

print(f"✅ 独立图像已保存：shap_force_plot_patient_*.png (SHAP 值已嵌入特征名)")